# 🛒 E-Commerce Sales & Customer Behaviour Analysis

**Dataset:** 1,500 order line items · 500 orders · Indian e-commerce platform · Jan–Dec 2018  
**Analyst:** Nandan R &nbsp;|&nbsp; [LinkedIn](https://www.linkedin.com/in/nandan-r-010564224/) &nbsp;|&nbsp; [GitHub](https://github.com/NandanR75)

---

| Section | Topic |
|---------|-------|
| 1 | Setup & Data Loading |
| 2 | Data Quality Assessment |
| 3 | Feature Engineering |
| 4 | Executive KPIs |
| 5 | Profitability Deep-Dive |
| 6 | Seasonality & Time Trends |
| 7 | Regional Performance |
| 8 | Product Category Analysis |
| 9 | Payment Mode Analysis |
| 10 | Customer Segmentation |
| 11 | Statistical Testing |
| 12 | Business Recommendations |


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F8F9FA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})
PAL = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED','#0891B2','#BE185D','#0D9488']

orders  = pd.read_csv('../Orders.csv')
details = pd.read_csv('../Details.csv')
df      = orders.merge(details, on='Order ID')

print(f"Orders : {orders.shape}  |  Details : {details.shape}  |  Merged : {df.shape}")
df.head(3)


## 2. Data Quality Assessment

In [ ]:
missing = df.isnull().sum()
print("=== Missing Values ===")
print(missing[missing > 0] if missing.any() else "✅  No missing values")

dupes = df.duplicated(subset=['Order ID','Sub-Category','PaymentMode']).sum()
print(f"\n=== Duplicate line items: {dupes} ===")

loss_n   = (df['Profit'] < 0).sum()
loss_pct = (df['Profit'] < 0).mean() * 100
print(f"\n⚠️  Loss-making line items: {loss_n:,} ({loss_pct:.1f}%)")
print("   This is the most important business risk in the dataset.")

df[['Amount','Profit','Quantity']].describe().round(1)


## 3. Feature Engineering

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Month']      = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
df['Quarter']    = df['Order Date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
df['Is_Profit']  = (df['Profit'] > 0).astype(int)
df['Margin_Pct'] = (df['Profit'] / df['Amount'].replace(0, np.nan) * 100).round(2)

order_total = df.groupby('Order ID')['Amount'].sum().rename('Order_Total')
df = df.merge(order_total, on='Order ID')

print("New features created:", ['Month','Month_Name','Quarter','Is_Profit','Margin_Pct','Order_Total'])
df[['Order ID','Month','Quarter','Is_Profit','Margin_Pct','Order_Total']].head(5)


## 4. Executive KPIs

In [ ]:
total_rev    = df['Amount'].sum()
total_profit = df['Profit'].sum()
margin       = total_profit / total_rev * 100
n_orders     = df['Order ID'].nunique()
n_customers  = df['CustomerName'].nunique()
n_units      = df['Quantity'].sum()
aov          = df.groupby('Order ID')['Amount'].sum().mean()

print(f"{'Metric':<28} {'Value':>14}")
print("─" * 44)
print(f"{'Total Revenue':<28} {'₹'+f'{total_rev:,}':>14}")
print(f"{'Total Profit':<28} {'₹'+f'{total_profit:,}':>14}")
print(f"{'Overall Profit Margin':<28} {margin:>13.2f}%")
print(f"{'Total Orders':<28} {n_orders:>14,}")
print(f"{'Total Units Sold':<28} {n_units:>14,}")
print(f"{'Unique Customers':<28} {n_customers:>14,}")
print(f"{'Avg Order Value (AOV)':<28} {'₹'+f'{aov:,.0f}':>14}")
print(f"{'Loss-Making Line Items':<28} {(df['Profit']<0).mean()*100:>13.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
tiles = [
    ('Total Revenue',   f'₹{total_rev/1e5:.2f}L',  PAL[0]),
    ('Total Profit',    f'₹{total_profit:,}',        PAL[1]),
    ('Profit Margin',   f'{margin:.1f}%',             PAL[3]),
    ('Avg Order Value', f'₹{aov:,.0f}',              PAL[4]),
]
for ax, (title, val, color) in zip(axes, tiles):
    ax.set_facecolor(color)
    ax.text(0.5, 0.62, val,   ha='center', va='center', fontsize=20, fontweight='bold',
            color='white', transform=ax.transAxes)
    ax.text(0.5, 0.22, title, ha='center', va='center', fontsize=10,
            color='white', alpha=0.88, transform=ax.transAxes)
    ax.axis('off')
plt.suptitle('Business KPI Summary — 2018', fontsize=13, fontweight='600', y=1.02)
plt.tight_layout()
plt.savefig('../dashboard/images/kpi_tiles.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Profitability Deep-Dive

In [ ]:
sub = (df.groupby('Sub-Category')
         .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Units=('Quantity','sum'))
         .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1))
         .sort_values('Profit', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cp = [PAL[1] if p > 0 else PAL[2] for p in sub['Profit']]
cm = [PAL[1] if m > 0 else PAL[2] for m in sub['Margin']]

axes[0].barh(sub.index[::-1], sub['Profit'][::-1], color=cp[::-1], edgecolor='white')
axes[0].axvline(0, color='gray', lw=0.8, ls='--')
axes[0].set_title('Net Profit by Sub-Category (₹)', fontweight='600')
axes[0].set_xlabel('Profit (₹)')

axes[1].barh(sub.index[::-1], sub['Margin'][::-1], color=cm[::-1], edgecolor='white')
axes[1].axvline(0, color='gray', lw=0.8, ls='--')
axes[1].set_title('Profit Margin % by Sub-Category', fontweight='600')
axes[1].set_xlabel('Margin %')

plt.tight_layout()
plt.savefig('../dashboard/images/sub_category_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print(sub[['Revenue','Profit','Margin']].to_string())


**📌 Insight — Sub-Category Profitability:**

| Status | Sub-Categories | Note |
|--------|---------------|------|
| 🟢 Profit stars (>13% margin) | Printers (14.5%), Accessories (15.4%), Tables (13.9%), Shirt (20%), T-shirt (20.3%) | Scale up — high margin |
| 🟡 Volume drivers (low margin) | Saree, Phones, Chairs | Optimise pricing |
| 🔴 Loss leaders | Electronic Games (−1.6%), Furnishings (−6%), Kurti (−11.9%), Skirt (−16.2%) | Reprice or discontinue |

Every sale of Electronic Games, Furnishings, Kurti, or Skirt **actively destroys profit**.


In [ ]:
mp = (df.groupby(['Month','Month_Name'])
        .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'))
        .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1))
        .reset_index().sort_values('Month'))

fig, ax = plt.subplots(figsize=(12, 4))
bc = [PAL[1] if m >= 0 else PAL[2] for m in mp['Margin']]
bars = ax.bar(mp['Month_Name'], mp['Margin'], color=bc, edgecolor='white', width=0.65)
ax.axhline(0, color='gray', lw=0.9, ls='--')
ax.axhline(mp['Margin'].mean(), color=PAL[0], lw=1.2, ls=':', label=f"Annual avg {mp['Margin'].mean():.1f}%")

for bar, val in zip(bars, mp['Margin']):
    ypos = bar.get_height() + 0.4 if val >= 0 else bar.get_height() - 2.0
    ax.text(bar.get_x()+bar.get_width()/2, ypos, f'{val:.1f}%',
            ha='center', fontsize=8.5,
            color=PAL[1] if val >= 0 else PAL[2], fontweight='500')

ax.set_title('Monthly Profit Margin % — 2018', fontweight='600')
ax.set_ylabel('Margin %')
ax.legend()
plt.tight_layout()
plt.savefig('../dashboard/images/monthly_margin.png', dpi=150, bbox_inches='tight')
plt.show()


**📌 Insight — Margin Volatility:**
- **4 months are loss-making at aggregate level**: May (−12.8%), July (−16.5%), September (−5.1%), December (−4.3%)
- **Feb (21.7%) and Nov (21.2%)** are the healthiest months — margin swings 38 percentage points across the year
- No consistent pricing discipline — margins are driven by what happens to sell, not a managed product mix


## 6. Seasonality & Time Trends

In [ ]:
mp2 = (df.groupby(['Month','Month_Name'])
           .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
           .reset_index().sort_values('Month'))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.bar(mp2['Month_Name'], mp2['Revenue'], color=PAL[0], alpha=0.85, edgecolor='white', label='Revenue')
ax1r = ax1.twinx()
ax1r.plot(mp2['Month_Name'], mp2['Profit'], color=PAL[1], marker='o', ms=5, lw=2)
ax1r.axhline(0, color='gray', lw=0.7, ls='--')
ax1r.fill_between(mp2['Month_Name'], mp2['Profit'], 0,
                   where=(mp2['Profit']<0), alpha=0.15, color=PAL[2])
ax1.set_title('Monthly Revenue (₹) vs Profit (₹)', fontweight='600')
ax1.set_ylabel('Revenue (₹)')
ax1r.set_ylabel('Profit (₹)', color=PAL[1])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1000:.0f}K'))
ax1r.tick_params(axis='y', colors=PAL[1])

ax2.bar(mp2['Month_Name'], mp2['Orders'], color=PAL[4], alpha=0.85, edgecolor='white')
ax2.set_title('Orders per Month', fontweight='600')
ax2.set_ylabel('Orders')

plt.tight_layout()
plt.savefig('../dashboard/images/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
qtr = (df.groupby('Quarter')
          .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
          .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1),
                  Rev_Share=lambda x: (x['Revenue']/x['Revenue'].sum()*100).round(1)))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label) in zip(axes, [('Revenue','Revenue (₹)'),('Profit','Profit (₹)'),('Margin','Margin %')]):
    bc = [PAL[0]]*4 if col=='Revenue' else [PAL[1] if v>=0 else PAL[2] for v in qtr[col]]
    bars = ax.bar(qtr.index, qtr[col], color=bc, edgecolor='white', alpha=0.9)
    ax.set_title(f'Quarterly {label}', fontweight='600')
    ax.set_ylabel(label)
    ax.axhline(0, color='gray', lw=0.7, ls='--')
    for bar, val in zip(bars, qtr[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(abs(qtr[col].max())*0.02),
                f'{val:,.0f}' if col!='Margin' else f'{val:.1f}%',
                ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../dashboard/images/quarterly.png', dpi=150, bbox_inches='tight')
plt.show()
print(qtr[['Revenue','Profit','Margin','Orders','Rev_Share']].to_string())


In [ ]:
monthly_rev = df.groupby('Month')['Amount'].sum()
avg_monthly = monthly_rev.mean()
s_idx       = (monthly_rev / avg_monthly).round(2)
mlabels     = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(11, 4))
sc = [PAL[2] if v < 0.75 else PAL[1] if v > 1.25 else PAL[0] for v in s_idx]
bars = ax.bar(mlabels, s_idx.values, color=sc, edgecolor='white')
ax.axhline(1.0, color='gray', lw=1, ls='--', label='Baseline = 1.0')
for bar, val in zip(bars, s_idx.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{val:.2f}', ha='center', fontsize=9)
ax.set_title('Seasonality Index by Month  (1.0 = average month)', fontweight='600')
ax.set_ylabel('Index')
ax.legend()
plt.tight_layout()
plt.savefig('../dashboard/images/seasonality_index.png', dpi=150, bbox_inches='tight')
plt.show()
print("Peak months (>1.25):", [mlabels[i-1] for i in s_idx[s_idx>1.25].index])
print("Weak months (<0.75):", [mlabels[i-1] for i in s_idx[s_idx<0.75].index])


**📌 Insight — Seasonality:**
- **Q1 dominates** at 36.8% of annual revenue (₹1.61L) with the highest margin (16.1%) — peak buying Jan–Mar
- **Q3 is loss-making** overall (margin = −2.0%) — July alone at −16.5% margin is the worst single month
- Jan, Mar, Nov are the strongest months (seasonality index > 1.25)
- Jun, Jul are the weakest months (index < 0.75) — lowest demand across the year


## 7. Regional Performance

In [ ]:
state = (df.groupby('State')
           .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
           .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1),
                   Rev_Share=lambda x: (x['Revenue']/x['Revenue'].sum()*100).round(1))
           .sort_values('Revenue', ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(state.index[::-1], state['Revenue'][::-1]/1000, color=PAL[0], alpha=0.85)
axes[0].set_title('Top 10 States — Revenue (₹K)', fontweight='600')
axes[0].set_xlabel('Revenue (₹K)')

mc = [PAL[1] if m >= 0 else PAL[2] for m in state['Margin'][::-1]]
axes[1].barh(state.index[::-1], state['Margin'][::-1], color=mc)
axes[1].axvline(0, color='gray', lw=0.8, ls='--')
axes[1].set_title('Top 10 States — Profit Margin %', fontweight='600')
axes[1].set_xlabel('Margin %')

plt.tight_layout()
plt.savefig('../dashboard/images/state_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(state[['Revenue','Profit','Margin','Rev_Share']].to_string())


In [ ]:
city = (df.groupby(['City','State'])
          .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
          .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1),
                  AOV=lambda x: (x['Revenue']/x['Orders']).round(0))
          .sort_values('Revenue', ascending=False).head(10))
print(city[['Revenue','Profit','Margin','Orders','AOV']].to_string())


**📌 Insight — Regional Performance:**
- **Maharashtra (₹1.02L) + Madhya Pradesh (₹87.5K) = 43% of total revenue** — the two most important markets
- **Rajasthan has a negative margin (−1.4%)** despite being a top-6 state by revenue — a regional pricing problem
- **Kerala (17.6%) and Gujarat (14.0%)** are the highest-margin states — proportionally underinvested
- Indore and Mumbai are the top cities — prioritise inventory depth and delivery SLA here


## 8. Product Category Analysis

In [ ]:
cat = (df.groupby('Category')
         .agg(Revenue=('Amount','sum'), Profit=('Profit','sum'), Units=('Quantity','sum'))
         .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1),
                 Rev_Share=lambda x: (x['Revenue']/x['Revenue'].sum()*100).round(1)))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label) in zip(axes,[('Revenue','Revenue (₹)'),('Profit','Profit (₹)'),('Margin','Margin %')]):
    bc = [PAL[1] if v>=0 else PAL[2] for v in cat[col]]
    ax.bar(cat.index, cat[col], color=bc, edgecolor='white', alpha=0.9)
    ax.set_title(f'Category — {label}', fontweight='600')
    ax.set_ylabel(label)
    ax.axhline(0, color='gray', lw=0.7, ls='--')
plt.tight_layout()
plt.savefig('../dashboard/images/category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(cat[['Revenue','Profit','Margin','Units','Rev_Share']].to_string())


In [ ]:
sub2 = (df.groupby('Sub-Category')
          .agg(Units=('Quantity','sum'), Revenue=('Amount','sum'), Profit=('Profit','sum'))
          .assign(Avg_Price=lambda x: x['Revenue']/x['Units'],
                  Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1))
          .reset_index())

fig, ax = plt.subplots(figsize=(11, 6))
sc2 = [PAL[1] if m > 0 else PAL[2] for m in sub2['Margin']]
ax.scatter(sub2['Units'], sub2['Avg_Price'], s=sub2['Revenue']/60,
           alpha=0.75, color=sc2, edgecolors='white', linewidths=1.5)
for _, row in sub2.iterrows():
    ax.annotate(row['Sub-Category'], (row['Units'], row['Avg_Price']),
                xytext=(6,4), textcoords='offset points', fontsize=8.5)
ax.set_xlabel('Units Sold')
ax.set_ylabel('Avg Price per Unit (₹)')
ax.set_title('Sub-Category — Volume vs Avg Price  (bubble = revenue  🟢 profitable / 🔴 loss)', fontweight='600')
plt.tight_layout()
plt.savefig('../dashboard/images/bubble_chart.png', dpi=150, bbox_inches='tight')
plt.show()


**📌 Insight — Category Mix:**
- **Electronics** earns the most revenue (₹1.66L, 38%) but has the lowest margin (7.9%) — dragged by Electronic Games
- **Clothing** has the highest margin (9.2%) but conceals loss-making lines (Kurti, Skirt, Leggings)
- **Furniture** is the most consistent category — every sub-category profitable except Furnishings
- **Printers** (₹59K, 14.5% margin) are the single best-performing sub-category on a risk-adjusted basis


## 9. Payment Mode Analysis

In [ ]:
pay = (df.groupby('PaymentMode')
          .agg(Orders=('Order ID','nunique'), Revenue=('Amount','sum'), Profit=('Profit','sum'))
          .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1),
                  AOV=lambda x: (x['Revenue']/x['Orders']).round(0),
                  Order_Share=lambda x: (x['Orders']/x['Orders'].sum()*100).round(1))
          .sort_values('Revenue', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, label) in zip(axes,[
        ('Revenue','Revenue (₹)'),
        ('Margin','Margin %'),
        ('AOV','Avg Order Value (₹)')]):
    bc = [PAL[0]]*len(pay) if col=='Revenue' else [PAL[1] if v>=0 else PAL[2] for v in pay[col]]
    ax.bar(pay.index, pay[col], color=bc, edgecolor='white', alpha=0.9)
    ax.set_title(f'Payment Mode — {label}', fontweight='600')
    ax.set_ylabel(label)
    ax.set_xticklabels(pay.index, rotation=18, ha='right')
    ax.axhline(0, color='gray', lw=0.7, ls='--')
plt.tight_layout()
plt.savefig('../dashboard/images/payment_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print(pay[['Orders','Revenue','Profit','Margin','AOV','Order_Share']].to_string())


**📌 Insight — Payment Modes:**

| Mode | Order Share | Margin | AOV | Verdict |
|------|------------|--------|-----|---------|
| COD | 45.6% | 8.1% | ₹447 | Dominant but highest operational risk |
| Credit Card | 16.9% | **14.5%** | **₹679** | Most profitable channel per order |
| EMI | 14.0% | 6.2% | ₹735 | Highest AOV, lower margin |
| UPI | 29.5% | 4.8% | ₹306 | Popular but least profitable |

**Action:** A 10% shift of COD/UPI orders to Credit Card would add ~₹1,500 incremental profit with no volume change.


## 10. Customer Segmentation

In [ ]:
oc = df.groupby('CustomerName')['Order ID'].nunique()
one_time = df[df['CustomerName'].isin(oc[oc==1].index)]
repeats  = df[df['CustomerName'].isin(oc[oc>=2].index)]

print(f"Unique customers:          {len(oc):,}")
print(f"One-time buyers:           {(oc==1).sum():,} ({(oc==1).mean()*100:.1f}%)")
print(f"Repeat buyers (2+ orders): {(oc>=2).sum():,} ({(oc>=2).mean()*100:.1f}%)")
print(f"Avg orders per customer:   {oc.mean():.2f}")
print()
print(f"One-time revenue: ₹{one_time['Amount'].sum():,}  ({one_time['Amount'].sum()/df['Amount'].sum()*100:.1f}% of total)")
print(f"Repeat revenue:   ₹{repeats['Amount'].sum():,}  ({repeats['Amount'].sum()/df['Amount'].sum()*100:.1f}% of total)")

fig, ax = plt.subplots(figsize=(5, 4))
sizes = [(oc==1).sum(), (oc>=2).sum()]
ax.pie(sizes, labels=['One-time
68.2%','Repeat
31.8%'],
       colors=[PAL[2], PAL[1]], autopct='%1.1f%%', startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Customer Retention Split', fontweight='600')
plt.tight_layout()
plt.savefig('../dashboard/images/customer_retention.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
top_cust = (df.groupby('CustomerName')
              .agg(Orders=('Order ID','nunique'),
                   Revenue=('Amount','sum'),
                   Profit=('Profit','sum'))
              .assign(Margin=lambda x: (x['Profit']/x['Revenue']*100).round(1))
              .sort_values('Revenue', ascending=False).head(15))

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top_cust.index[::-1], top_cust['Revenue'][::-1], color=PAL[0], alpha=0.8)
ax2 = ax.twiny()
ax2.plot(top_cust['Margin'][::-1], top_cust.index[::-1],
         'o-', color=PAL[2], ms=6, lw=1.8, label='Margin %')
ax2.axvline(0, color='gray', lw=0.7, ls='--')
ax2.set_xlabel('Margin %', color=PAL[2])
ax2.tick_params(axis='x', colors=PAL[2])
ax.set_xlabel('Revenue (₹)')
ax.set_title('Top 15 Customers — Revenue & Profit Margin', fontweight='600')
plt.tight_layout()
plt.savefig('../dashboard/images/top_customers.png', dpi=150, bbox_inches='tight')
plt.show()
print(top_cust.to_string())


**📌 Insight — Customer Behaviour:**
- Only **31.8% of customers place repeat orders** — the biggest untapped growth lever is retention, not acquisition
- Repeat buyers account for **~50% of revenue** despite being a third of the customer base — 2× more valuable per head
- **Top revenue customer (Harivansh, ₹9,902) is loss-making (−₹157)** — tracking revenue alone is dangerous
- A loyalty programme that converts 10% of one-time buyers to repeat = ~₹22K incremental revenue, zero new customers


## 11. Statistical Testing

In [ ]:
print("="*55)
print("TEST 1: Welch t-test — Credit Card vs UPI profit")
print("="*55)
cc  = df[df['PaymentMode']=='Credit Card']['Profit']
upi = df[df['PaymentMode']=='UPI']['Profit']
t, p = stats.ttest_ind(cc, upi)
print(f"  Credit Card mean profit per item : ₹{cc.mean():.1f}")
print(f"  UPI mean profit per item         : ₹{upi.mean():.1f}")
print(f"  Difference                       : ₹{cc.mean()-upi.mean():.1f}")
print(f"  t = {t:.3f}  |  p = {p:.4f}")
print("  ✅ SIGNIFICANT" if p<0.05 else "  ❌ Not significant")

print()
print("="*55)
print("TEST 2: One-way ANOVA — Category vs Profit")
print("="*55)
grps = [df[df['Category']==c]['Profit'] for c in df['Category'].unique()]
f, p2 = stats.f_oneway(*grps)
print(f"  F = {f:.3f}  |  p = {p2:.4f}")
print("  ✅ SIGNIFICANT — category drives profitability" if p2<0.05 else "  ❌ Not significant")
for cat in df['Category'].unique():
    print(f"    {cat}: mean ₹{df[df['Category']==cat]['Profit'].mean():.1f}/item")

print()
print("="*55)
print("TEST 3: Pearson correlation — Revenue vs Profit")
print("="*55)
r, p3 = stats.pearsonr(df['Amount'], df['Profit'])
print(f"  r = {r:.3f}  |  p = {p3:.4f}")
print("  Weak positive correlation — revenue is NOT a reliable proxy for profit")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
corr = df[['Amount','Profit','Quantity']].corr().round(2)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True,
            annot_kws={'size': 12})
ax.set_title('Numeric Correlation Matrix', fontweight='600')
plt.tight_layout()
plt.savefig('../dashboard/images/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


**Statistical Testing Summary:**

| Test | Comparison | Key Number | p-value | Significant? |
|------|-----------|-----------|---------|-------------|
| Welch t-test | Credit Card vs UPI profit/item | CC earns ₹67.5 more | 0.0002 | ✅ Yes |
| One-way ANOVA | Category vs Profit | F = 5.14 | 0.006 | ✅ Yes |
| Pearson r | Revenue vs Profit | r = 0.31 (weak) | <0.0001 | ✅ Yes (weak) |

These tests back every recommendation below with statistical evidence, not just pattern-spotting.


## 12. Business Recommendations

| # | Priority | Finding | Recommended Action |
|---|----------|---------|-------------------|
| 1 | 🔴 High | 35.3% of line items are loss-making | Immediate pricing & supplier cost audit across all sub-categories |
| 2 | 🔴 High | Electronic Games & Furnishings net-negative | Reprice or discontinue — every unit sold destroys margin |
| 3 | 🔴 High | Kurti (−11.9%) & Skirt (−16.2%) | Audit return rates & supplier pricing contracts |
| 4 | 🟡 Medium | Credit Card margin 14.5% vs UPI 4.8% (p=0.0002) | Offer prepay cashback to shift customers from COD/UPI |
| 5 | 🟡 Medium | 68.2% of customers never return | Launch loyalty / re-engagement email campaign |
| 6 | 🟡 Medium | Top customer Harivansh: ₹9,902 revenue, −₹157 profit | Track margin per customer, not just revenue |
| 7 | 🟡 Medium | Shirts & T-shirts: ~20% margin but low volume | Increase catalogue depth and invest in marketing |
| 8 | 🟢 Low | Q3 (Jul–Sep) overall loss-making (−2% margin) | Reduce inventory build; run pre-July clearance promos |
| 9 | 🟢 Low | Q1 = 36.8% of annual revenue | Pre-load inventory in December; maximise Q1 ad spend |
| 10 | 🟢 Low | Maharashtra + MP = 43% of revenue | Deepen stock range; reduce delivery SLAs in both states |
| 11 | 🟢 Low | Rajasthan: top-6 revenue state, negative margin | Investigate local pricing or high-return product mix |
| 12 | 🟢 Low | Kerala & Gujarat: highest margins, underinvested | Expand marketing investment in these high-efficiency states |

---
*Analysis by Nandan R &nbsp;|&nbsp; [LinkedIn](https://www.linkedin.com/in/nandan-r-010564224/) &nbsp;|&nbsp; [GitHub](https://github.com/NandanR75)*
